# CNN + Shifter on **Mouse** data (vs Sensorium comparison)

Same model as `train_cnn_shifter_epoch_vs_score` but with full metrics as in `train_cnn_shifter_table_1_sensorium`:
- Valid-neuron filtering, per-neuron correlation/R²/EV, all epoch-vs-* saves, per-neuron pkl.

We train **multiple models** varying only **max training sample size** (e.g. 500, 1000, 2000, 5000, 10000, full) to compare with Sensorium (~568 train). Saves include `dataset="mouse"` and `max_train_samples` so `plot_epoch_vs_all_metrics` can plot Sensorium vs Mouse comparison.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Subset, DataLoader, ConcatDataset
from mouse_model.data_utils_new import MouseDatasetSegNewBehav
import numpy as np
from mouse_model.evaluation import cor_in_time
from sklearn.metrics import r2_score, mean_squared_error
import random, os
import pickle
from kornia.geometry.transform import get_affine_matrix2d, warp_affine

In [ ]:
class Shifter(nn.Module):
    def __init__(self, input_dim=4, output_dim=3, hidden_dim=256):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.layers = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim),
            nn.Tanh(),
        )
        self.bias = nn.Parameter(torch.zeros(3))
    def forward(self, x):
        x = x.reshape(-1, self.input_dim)
        x = self.layers(x)
        x0 = (x[...,0] + self.bias[0]) * 80/4
        x1 = (x[...,1] + self.bias[1]) * 60/4
        x2 = (x[...,2] + self.bias[2]) * 180/4
        x = torch.stack([x0, x1, x2], dim=-1)
        x = x.reshape(-1, 1, self.output_dim)
        return x

In [ ]:
def size_helper(in_length, kernel_size, padding=0, dilation=1, stride=1):
    res = in_length + 2 * padding - dilation * (kernel_size - 1) - 1
    res /= stride
    res += 1
    return np.floor(res)

class VisualEncoder(nn.Module):
    def __init__(self, output_dim, input_shape=(60, 80), k1=7, k2=7, k3=7):
        super().__init__()
        self.input_shape = (60, 80)
        out_shape_0 = size_helper(in_length=input_shape[0], kernel_size=k1, stride=2)
        out_shape_0 = size_helper(in_length=out_shape_0, kernel_size=k2, stride=2)
        out_shape_0 = size_helper(in_length=out_shape_0, kernel_size=k3, stride=2)
        out_shape_1 = size_helper(in_length=input_shape[1], kernel_size=k1, stride=2)
        out_shape_1 = size_helper(in_length=out_shape_1, kernel_size=k2, stride=2)
        out_shape_1 = size_helper(in_length=out_shape_1, kernel_size=k3, stride=2)
        self.output_shape = (int(out_shape_0), int(out_shape_1))
        self.layers = nn.Sequential(
            nn.Conv2d(1, 128, kernel_size=k1, stride=2),
            nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout(p=0.5),
            nn.Conv2d(128, 64, kernel_size=k2, stride=2),
            nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout(p=0.5),
            nn.Conv2d(64, 32, kernel_size=k3, stride=2),
            nn.BatchNorm2d(32), nn.ReLU(), nn.Dropout(p=0.5),
            nn.Flatten(),
            nn.Linear(self.output_shape[0]*self.output_shape[1]*32, output_dim),
        )
    def forward(self, x):
        return self.layers(x)

In [ ]:
class Predictor(nn.Module):
    def __init__(self, num_neurons):
        super().__init__()
        self.encoder = VisualEncoder(output_dim=num_neurons)
        self.softplus = nn.Softplus()
        self.shifter = Shifter()

    def forward(self, images, behav):
        if args.shifter:
            bs = images.size()[0]
            behav_shifter = torch.concat((behav[...,4].unsqueeze(-1), behav[...,3].unsqueeze(-1),
                                          behav[...,1].unsqueeze(-1), behav[...,2].unsqueeze(-1)), dim=-1)
            shift_param = self.shifter(behav_shifter)
            shift_param = shift_param.reshape(-1, 3)
            scale_param = torch.ones_like(shift_param[..., 0:2]).to(shift_param.device)
            affine_mat = get_affine_matrix2d(
                translations=shift_param[..., 0:2], scale=scale_param,
                center=torch.repeat_interleave(torch.tensor([[30, 40]], dtype=torch.float), bs*1, dim=0).to(shift_param.device),
                angle=shift_param[..., 2])
            affine_mat = affine_mat[:, :2, :]
            images = warp_affine(images, affine_mat, dsize=(60, 80))
        pred = self.encoder(images)
        pred = self.softplus(pred)
        return pred

In [ ]:
class Args:
    seed = 0
    file_id = "070921_J553RT"
    epochs = 100
    batch_size = 256
    seq_len = 1
    num_neurons = None  # set from dataset (MouseDatasetSegNewBehav drops bad neurons -> 68 for 070921_J553RT)
    learning_rate = 0.0001
    segment_num = 10
    vid_type = "vid_mean"
    max_train_samples = None  # None = use full train; int = cap (for comparison with Sensorium ~568)
    best_train_path = None
    best_val_path = None
    shifter = False

args = Args()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(args.seed)
os.environ["PYTHONHASHSEED"] = str(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.cuda.manual_seed(args.seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("CUDA:", torch.cuda.is_available())

CUDA: True


In [ ]:
def load_train_val_ds():
    ds_list = [MouseDatasetSegNewBehav(file_id=args.file_id, segment_num=args.segment_num, seg_idx=i, data_split="train",
                                       vid_type=args.vid_type, seq_len=args.seq_len, predict_offset=1)
               for i in range(args.segment_num)]
    train_ds, val_ds = [], []
    for ds in ds_list:
        train_ratio = 0.8
        train_ds_len = int(len(ds) * train_ratio)
        train_ds.append(Subset(ds, np.arange(0, train_ds_len, 1)))
        val_ds.append(Subset(ds, np.arange(train_ds_len, len(ds), 1)))
    train_ds = ConcatDataset(train_ds)
    val_ds = ConcatDataset(val_ds)
    if args.max_train_samples is not None:
        n_total = len(train_ds)
        n_use = min(args.max_train_samples, n_total)
        perm = np.random.RandomState(args.seed).permutation(n_total)[:n_use]
        train_ds = Subset(train_ds, perm)
        print("max_train_samples={} -> using {} train (val unchanged {})".format(args.max_train_samples, n_use, len(val_ds)))
    else:
        print("Full train size: {} val: {}".format(len(train_ds), len(val_ds)))
    return train_ds, val_ds

In [ ]:
def load_test_ds():
    test_ds = [MouseDatasetSegNewBehav(file_id=args.file_id, segment_num=args.segment_num, seg_idx=i, data_split="test",
                                       vid_type=args.vid_type, seq_len=args.seq_len, predict_offset=1)
               for i in range(args.segment_num)]
    test_ds = ConcatDataset(test_ds)
    return test_ds

In [ ]:
def train_model():
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(args.seed)
    train_ds, val_ds = load_train_val_ds()
    train_dataloader = DataLoader(dataset=train_ds, batch_size=args.batch_size, shuffle=True, num_workers=8)
    val_dataloader = DataLoader(dataset=val_ds, batch_size=args.batch_size, shuffle=False, num_workers=8)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate)
    best_train_loss = np.inf
    best_val_loss = np.inf
    train_loss_list = []
    val_loss_list = []
    val_cor_list = []
    val_r2_list = []
    val_mse_list = []
    val_poisson_loss_list = []
    val_bits_per_spike_list = []
    val_explained_var_list = []
    cor_per_neuron_per_epoch = []
    r2_per_neuron_per_epoch = []
    ev_per_neuron_per_epoch = []
    n_valid_per_epoch = []
    ct = 0
    for epoch in range(args.epochs):
        print("Start epoch", epoch)
        model.train()
        epoch_train_loss = 0
        for (image, behav, spikes) in train_dataloader:
            image, behav, spikes = image.to(device), behav.to(device), spikes.to(device)
            image = torch.squeeze(image, axis=1)
            pred = model(image, behav)
            loss = nn.functional.poisson_nll_loss(pred, spikes, reduction="mean", log_input=False)
            epoch_train_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        epoch_train_loss = epoch_train_loss / len(train_dataloader)
        train_loss_list.append(epoch_train_loss)
        if epoch_train_loss < best_train_loss:
            torch.save(model.state_dict(), args.best_train_path)
            best_train_loss = epoch_train_loss
            if len(val_dataloader) == 0:
                torch.save(model.state_dict(), args.best_val_path)
        print("Epoch {} train loss: {:.4f}".format(epoch, epoch_train_loss))
        model.eval()
        epoch_val_loss = 0
        pred_val_all = []
        label_val_all = []
        if len(val_dataloader) > 0:
            with torch.no_grad():
                for (image, behav, spikes) in val_dataloader:
                    image, behav, spikes = image.to(device), behav.to(device), spikes.to(device)
                    image = torch.squeeze(image, axis=1)
                    pred = model(image, behav)
                    loss = nn.functional.poisson_nll_loss(pred, spikes, reduction="mean", log_input=False)
                    epoch_val_loss += loss.item()
                    pred_val_all.append(pred.cpu().numpy())
                    label_val_all.append(spikes.cpu().numpy())
            epoch_val_loss = epoch_val_loss / len(val_dataloader)
            pred_val = np.concatenate(pred_val_all, axis=0)
            label_val = np.concatenate(label_val_all, axis=0)
            var_pred = np.var(pred_val, axis=0)
            var_label = np.var(label_val, axis=0)
            eps = 1e-12
            valid = (var_pred > eps) & (var_label > eps)
            n_valid = int(valid.sum())
            num_neurons = pred_val.shape[1]
            cor_array = cor_in_time(pred_val, label_val)
            cor_per_neuron = np.array(cor_array.flatten(), dtype=np.float64)
            cor_per_neuron[~valid] = np.nan
            mean_cor = np.nanmean(cor_per_neuron)
            if np.isnan(mean_cor):
                mean_cor = 0.0
            val_cor_list.append(mean_cor)
            cor_per_neuron_per_epoch.append(cor_per_neuron.copy())
            n_valid_per_epoch.append(n_valid)
            r2_per_neuron = np.array([r2_score(label_val[:, j], pred_val[:, j]) for j in range(num_neurons)], dtype=np.float64)
            r2_per_neuron[~valid] = np.nan
            mean_r2 = np.nanmean(r2_per_neuron)
            val_r2_list.append(float(mean_r2) if not np.isnan(mean_r2) else 0.0)
            r2_per_neuron_per_epoch.append(r2_per_neuron.copy())
            if n_valid > 0:
                mse = mean_squared_error(label_val[:, valid], pred_val[:, valid])
            else:
                mse = mean_squared_error(label_val, pred_val)
            val_mse_list.append(float(mse))
            val_poisson_loss_list.append(float(epoch_val_loss))
            val_bits_per_spike_list.append(float(epoch_val_loss / np.log(2)))
            res = label_val - pred_val
            var_res = np.var(res, axis=0)
            with np.errstate(divide="ignore", invalid="ignore"):
                ev_per_neuron = np.where(var_label > eps, 1.0 - var_res / var_label, np.nan).astype(np.float64)
            ev_per_neuron[~valid] = np.nan
            mean_ev = np.nanmean(ev_per_neuron)
            val_explained_var_list.append(float(mean_ev) if not np.isnan(mean_ev) else 0.0)
            ev_per_neuron_per_epoch.append(ev_per_neuron.copy())
        else:
            epoch_val_loss = np.inf
            val_cor_list.append(float("nan"))
            val_r2_list.append(float("nan"))
            val_mse_list.append(float("nan"))
            val_poisson_loss_list.append(float("nan"))
            val_bits_per_spike_list.append(float("nan"))
            val_explained_var_list.append(float("nan"))
            cor_per_neuron_per_epoch.append(None)
            r2_per_neuron_per_epoch.append(None)
            ev_per_neuron_per_epoch.append(None)
            n_valid_per_epoch.append(None)
        val_loss_list.append(epoch_val_loss)
        if epoch_val_loss < best_val_loss:
            torch.save(model.state_dict(), args.best_val_path)
            best_val_loss = epoch_val_loss
            ct = 0
        else:
            ct += 1
            if len(val_dataloader) > 0 and ct > 5:
                print("stop training")
                break
        if len(val_dataloader) > 0:
            print("Epoch {} val loss: {:.4f} | corr: {:.4f} R2: {:.4f} MSE: {:.4f} EV: {:.4f} | valid neurons: {} / {}".format(
                epoch, epoch_val_loss, mean_cor, val_r2_list[-1], val_mse_list[-1], val_explained_var_list[-1], n_valid, num_neurons))
        else:
            print("Epoch {} val loss: {}".format(epoch, epoch_val_loss))
        print("End epoch", epoch)
    return (train_loss_list, val_loss_list, val_cor_list, val_r2_list, val_mse_list,
            val_poisson_loss_list, val_bits_per_spike_list, val_explained_var_list,
            cor_per_neuron_per_epoch, r2_per_neuron_per_epoch, ev_per_neuron_per_epoch, n_valid_per_epoch)

In [ ]:
# Train multiple configs: vary max_train_samples (and shifter). Saves go to same dirs as Sensorium with dataset="mouse" and max_train_samples.
MAX_TRAIN_SAMPLES_LIST = [100,200,300,400,500,600,1000]  # None = full (~30120 for segment_num=10)
# Optionally reduce for a quick test: MAX_TRAIN_SAMPLES_LIST = [500, None]

file_id = args.file_id
vid_type = args.vid_type
segment_num = args.segment_num
base_dir = "/home/herbelinluke/Downloads/paths/mouse_vs_sensorium"
os.makedirs(base_dir, exist_ok=True)

# Get num_neurons from dataset (must match data: 68 for 070921_J553RT after bad-neuron drop)
_ds0 = MouseDatasetSegNewBehav(file_id=file_id, segment_num=segment_num, seg_idx=0, data_split="train",
                              vid_type=vid_type, seq_len=args.seq_len, predict_offset=1)
num_neurons = _ds0.nsp.shape[1]
args.num_neurons = num_neurons
print("num_neurons from dataset:", num_neurons)

for max_train_samples in MAX_TRAIN_SAMPLES_LIST:
    args.max_train_samples = max_train_samples
    for shifter in [False, True]:
        args.shifter = shifter
        args.best_train_path = os.path.join(base_dir, "train_cnn_mouse_{}_shifter_{}_maxsamp_{}.pth".format(
            file_id, shifter, max_train_samples if max_train_samples is not None else "full"))
        args.best_val_path = os.path.join(base_dir, "val_cnn_mouse_{}_shifter_{}_maxsamp_{}.pth".format(
            file_id, shifter, max_train_samples if max_train_samples is not None else "full"))
        print("--- max_train_samples={} shifter={} ---".format(max_train_samples, shifter))
        model = Predictor(num_neurons=num_neurons).to(device)
        (train_loss_list, val_loss_list, val_cor_list, val_r2_list, val_mse_list,
         val_poisson_loss_list, val_bits_per_spike_list, val_explained_var_list,
         cor_per_neuron_per_epoch, r2_per_neuron_per_epoch, ev_per_neuron_per_epoch, n_valid_per_epoch) = train_model()
        maxsamp_str = "full" if max_train_samples is None else max_train_samples
        base_meta = {
            "model_name": "cnn",
            "file_id": file_id,
            "vid_type": vid_type,
            "shifter": shifter,
            "dataset": "mouse",
            "max_train_samples": max_train_samples,
            "segment_num": segment_num,
            "train_loss_list": train_loss_list,
            "val_loss_list": val_loss_list,
        }
        fname = "epoch_vs_score_cnn_mouse_{}_{}_shifter_{}_maxsamp_{}.pkl".format(file_id, vid_type, shifter, maxsamp_str)
        score_dirs = [
            ("epoch_vs_score_data", "val_cor_list", val_cor_list),
            ("epoch_vs_correlation_data", "val_cor_list", val_cor_list),
            ("epoch_vs_r2_data", "val_r2_list", val_r2_list),
            ("epoch_vs_mse_data", "val_mse_list", val_mse_list),
            ("epoch_vs_poisson_loss_data", "val_poisson_loss_list", val_poisson_loss_list),
            ("epoch_vs_bits_per_spike_data", "val_bits_per_spike_list", val_bits_per_spike_list),
            ("epoch_vs_explained_variance_data", "val_explained_var_list", val_explained_var_list),
        ]
        for dir_name, score_key, score_list in score_dirs:
            os.makedirs(dir_name, exist_ok=True)
            save_path = os.path.join(dir_name, fname)
            with open(save_path, "wb") as f:
                pickle.dump({**base_meta, score_key: score_list}, f)
            print("Saved to", save_path)
        per_neuron_dir = "epoch_vs_per_neuron_data"
        os.makedirs(per_neuron_dir, exist_ok=True)
        per_neuron_fname = "per_neuron_cnn_mouse_{}_{}_shifter_{}_maxsamp_{}.pkl".format(file_id, vid_type, shifter, maxsamp_str)
        per_neuron_path = os.path.join(per_neuron_dir, per_neuron_fname)
        with open(per_neuron_path, "wb") as f:
            pickle.dump({
                **base_meta,
                "cor_per_neuron_per_epoch": cor_per_neuron_per_epoch,
                "r2_per_neuron_per_epoch": r2_per_neuron_per_epoch,
                "ev_per_neuron_per_epoch": ev_per_neuron_per_epoch,
                "n_valid_per_epoch": n_valid_per_epoch,
            }, f)
        print("Saved to", per_neuron_path)

num_neurons from dataset: 68
--- max_train_samples=100 shifter=False ---
max_train_samples=100 -> using 100 train (val unchanged 7540)
Start epoch 0
Epoch 0 train loss: 0.9556
Epoch 0 val loss: 0.9043 | corr: 0.0057 R2: -0.5426 MSE: 1.0979 EV: 0.0000 | valid neurons: 68 / 68
End epoch 0
Start epoch 1
Epoch 1 train loss: 0.9343
Epoch 1 val loss: 0.9039 | corr: 0.0016 R2: -0.5406 MSE: 1.0973 EV: 0.0000 | valid neurons: 68 / 68
End epoch 1
Start epoch 2
Epoch 2 train loss: 0.9316
Epoch 2 val loss: 0.9038 | corr: -0.0029 R2: -0.5390 MSE: 1.0971 EV: -0.0000 | valid neurons: 68 / 68
End epoch 2
Start epoch 3
Epoch 3 train loss: 0.9127
Epoch 3 val loss: 0.9034 | corr: -0.0041 R2: -0.5366 MSE: 1.0966 EV: -0.0001 | valid neurons: 68 / 68
End epoch 3
Start epoch 4
Epoch 4 train loss: 0.8989
Epoch 4 val loss: 0.9027 | corr: -0.0045 R2: -0.5336 MSE: 1.0957 EV: -0.0001 | valid neurons: 68 / 68
End epoch 4
Start epoch 5
Epoch 5 train loss: 0.8977
Epoch 5 val loss: 0.9016 | corr: -0.0058 R2: -0.5296 